# FarmX Counter — train v0.2 (mạng + đảo màu + tôm vẽ tự sinh)
Colab T4, Chạy tất cả, ~80 phút. Model tự lên Supabase `dem_v02.onnx`.

In [ ]:
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="Go6DqabG3FqADGMkr49U")
ds = rf.workspace("nguyen-hai-dang-10csz").project("risishrimp-count-suuvb").version(1).download("yolov8")
ROOT = ds.location; print(ROOT)

## Đảo màu dataset mạng

In [ ]:
import os, glob, shutil
from PIL import Image, ImageOps
for split in ["train","valid"]:
    imd=f"{ROOT}/{split}/images"; lbd=f"{ROOT}/{split}/labels"; n=0
    for p in glob.glob(imd+"/*"):
        base,ext=os.path.splitext(os.path.basename(p))
        if base.endswith("_inv") or base.startswith("syn_"): continue
        ImageOps.invert(Image.open(p).convert("RGB")).save(f"{imd}/{base}_inv{ext}", quality=92)
        lb=f"{lbd}/{base}.txt"
        if os.path.exists(lb): shutil.copy(lb, f"{lbd}/{base}_inv.txt")
        n+=1
    print(split, n)

## Sinh ảnh tôm vẽ (giống tờ in) + nhãn trọn thân
Thân trong mờ, 2 mắt đen, gan cam; xoay ngẫu nhiên; nền trắng/xám nhẹ; mờ + nhiễu như giấy in.

In [ ]:
import random, math, numpy as np
from PIL import Image, ImageDraw, ImageFilter
def ve_tom(draw, cx, cy, L, ang):
    a=math.radians(ang); c,s=math.cos(a),math.sin(a)
    def P(x,y): return (cx+x*c-y*s, cy+x*s+y*c)
    body=[P(-L*0.5,0),P(-L*0.3,L*0.11),P(0,L*0.13),P(L*0.3,L*0.11),P(L*0.5,0),P(L*0.3,-L*0.11),P(0,-L*0.13),P(-L*0.3,-L*0.11)]
    g=random.randint(150,195); draw.polygon(body, fill=(g,g,g), outline=(g-40,g-40,g-40))
    draw.line([P(L*0.5,0),P(L*0.63,L*0.06)],fill=(120,120,120),width=1); draw.line([P(L*0.5,0),P(L*0.63,-L*0.06)],fill=(120,120,120),width=1)
    draw.line([P(-L*0.5,0),P(-L*0.78,L*0.09)],fill=(140,140,140),width=1); draw.line([P(-L*0.5,0),P(-L*0.78,-L*0.09)],fill=(140,140,140),width=1)
    gx,gy=P(-L*0.22,0); r=L*0.09; draw.ellipse([gx-r*1.3,gy-r*0.7,gx+r*1.3,gy+r*0.7], fill=(random.randint(200,240),random.randint(110,150),random.randint(20,60)))
    for sy in (L*0.06,-L*0.06):
        ex,ey=P(-L*0.43,sy); re=max(1.2,L*0.03); draw.ellipse([ex-re,ey-re,ex+re,ey+re], fill=(15,15,15))
    xs=[p[0] for p in body]+[P(-L*0.78,0)[0]]; ys=[p[1] for p in body]
    return min(xs),min(ys),max(xs),max(ys)
def sinh(path_img, path_lb, W=1280, H=960):
    bg=random.randint(200,250); img=Image.new("RGB",(W,H),(bg,bg,bg+random.randint(-5,5)))
    d=ImageDraw.Draw(img)
    n=random.choice([20,40,80,150,250,400,500]); L=random.uniform(28,70)
    pts=[]; lines=[]
    for _ in range(n*30):
        if len(pts)>=n: break
        x=random.uniform(L,W-L); y=random.uniform(L,H-L)
        if all(math.hypot(x-px,y-py)>L*random.uniform(0.45,0.8) for px,py in pts): pts.append((x,y))
    for x,y in pts:
        x0,y0,x1,y1=ve_tom(d,x,y,L*random.uniform(0.85,1.15),random.uniform(0,360))
        x0,x1=max(0,x0),min(W,x1); y0,y1=max(0,y0),min(H,y1)
        lines.append(f"0 {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
    img=img.filter(ImageFilter.GaussianBlur(random.uniform(0.4,1.6)))
    a=np.array(img).astype(np.int16)+np.random.normal(0,random.uniform(2,8),(H,W,1)).astype(np.int16)
    img=Image.fromarray(np.clip(a,0,255).astype(np.uint8))
    if random.random()<0.5: img=img.resize((640,480))
    img.save(path_img,quality=random.randint(70,92)); open(path_lb,"w").write("\n".join(lines))
random.seed(1)
for split,N in [("train",2600),("valid",300)]:
    for i in range(N): sinh(f"{ROOT}/{split}/images/syn_{i:05d}.jpg", f"{ROOT}/{split}/labels/syn_{i:05d}.txt")
print("da sinh 2900 anh")
Image.open(f"{ROOT}/train/images/syn_00003.jpg")

## Train

In [ ]:
from ultralytics import YOLO
m = YOLO("yolo11n.pt")
m.train(data=ROOT+"/data.yaml", epochs=40, imgsz=640, batch=32, degrees=180, flipud=0.5, fliplr=0.5, scale=0.4,
        hsv_v=0.6, hsv_s=0.5, mosaic=1.0, patience=12, project="farmx", name="dem_v02", exist_ok=True)

## Sai số đếm theo từng loại

In [ ]:
import glob, os, numpy as np
from ultralytics import YOLO
best = YOLO("/content/runs/detect/farmx/dem_v02/weights/best.pt")
def sai(files):
    s=[]
    for p in files[:120]:
        lb=p.replace("/images/","/labels/").rsplit(".",1)[0]+".txt"; that=sum(1 for _ in open(lb)) if os.path.exists(lb) else 0
        may=len(best.predict(p,imgsz=1280 if "syn_" in p else 640,conf=0.3,verbose=False)[0].boxes); s.append(abs(may-that)/max(that,1)*100)
    return np.mean(s)
V=ROOT+"/valid/images/"
print("nen toi  : %.1f%%" % sai([p for p in sorted(glob.glob(V+"*")) if "_inv" not in p and "syn_" not in p]))
print("nen sang : %.1f%%" % sai(sorted(glob.glob(V+"*_inv.jpg"))))
print("tom ve   : %.1f%%" % sai(sorted(glob.glob(V+"syn_*.jpg"))))

## Xuất ONNX → Supabase

In [ ]:
best.export(format="onnx", imgsz=1280, opset=12, simplify=True, dynamic=False)
import requests, os
F="/content/runs/detect/farmx/dem_v02/weights/best.onnx"; print(os.path.getsize(F))
K="sb_publishable_DeOQ4ZYgl_6Oxth4eYyrbg_EBiJ6unP"
U="https://xofhpbfiuolkcbwbxume.supabase.co/storage/v1/object/counter-model/dem_v02.onnx"
r=requests.post(U,headers={"apikey":K,"Authorization":"Bearer "+K,"Content-Type":"application/octet-stream","x-upsert":"true"},data=open(F,"rb").read())
print(r.status_code, r.text[:120])